# Download, Import, And Validate Data

This notebook downloads the raw data from Hugging Face, imports the raw parquet files, and validates the cleaned outputs produced by `clean_data.py`.

## 1. Imports And Paths

Install dependencies from `requirements.txt` before running this notebook:

```powershell
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import json

import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

REPO = "TheFinAI/Herculean"
DOWNLOAD_DIR = Path("data/raw")
RAW_DIR = DOWNLOAD_DIR / "data"
PROCESSED_DIR = Path("data/processed")

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

## 2. Download Raw Data

This downloads `prices.parquet`, `news.parquet`, and `filings.parquet` into `data/raw/data`.

In [ ]:
for name in ["prices", "news", "filings"]:
    hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=f"data/{name}.parquet",
        local_dir=DOWNLOAD_DIR,
    )

info = HfApi().dataset_info(REPO)

print("Dataset revision:", info.sha)
print("Last modified:", info.last_modified)
print("Download date:", pd.Timestamp.today().date())

## 3. Import Raw Data

In [ ]:
prices_raw = pd.read_parquet(RAW_DIR / "prices.parquet")
news_raw = pd.read_parquet(RAW_DIR / "news.parquet")
filings_raw = pd.read_parquet(RAW_DIR / "filings.parquet")

raw = {
    "prices": prices_raw,
    "news": news_raw,
    "filings": filings_raw,
}

pd.DataFrame([
    {"dataset": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in raw.items()
])

## 4. Raw Data Preview

In [ ]:
for name, df in raw.items():
    print(f"\n{name.upper()}")
    display(df.head())

## 5. Validate Cleaned Outputs

Run `python clean_data.py` before this section if the cleaned outputs do not exist yet.

In [ ]:
clean = {
    "prices": pd.read_parquet(PROCESSED_DIR / "prices_clean.parquet"),
    "news": pd.read_parquet(PROCESSED_DIR / "news_clean.parquet"),
    "filings": pd.read_parquet(PROCESSED_DIR / "filings_clean.parquet"),
    "dashboard": pd.read_parquet(PROCESSED_DIR / "dashboard_daily.parquet"),
}

pd.DataFrame([
    {"dataset": name, "rows": len(df), "columns": len(df.columns)}
    for name, df in clean.items()
])

### Row Counts Before And After

In [ ]:
row_counts = []

for name in ["prices", "news", "filings"]:
    row_counts.append({
        "dataset": name,
        "raw_rows": len(raw[name]),
        "clean_rows": len(clean[name]),
        "row_difference": len(clean[name]) - len(raw[name]),
    })

pd.DataFrame(row_counts)

### Missing Values Before And After

In [ ]:
def missing_summary(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    })

for name in ["prices", "news", "filings"]:
    print(f"\n{name.upper()} RAW")
    display(missing_summary(raw[name]))
    print(f"{name.upper()} CLEAN")
    display(missing_summary(clean[name]))

### Unique Values In Important Categorical Columns

In [ ]:
categorical_checks = [
    ("prices", "symbol"),
    ("news", "symbol"),
    ("filings", "symbol"),
    ("filings", "document_type"),
]

for dataset, column in categorical_checks:
    print(f"\n{dataset}.{column} raw")
    display(raw[dataset][column].value_counts(dropna=False).sort_index())
    print(f"{dataset}.{column} clean")
    display(clean[dataset][column].value_counts(dropna=False).sort_index())

### Min, Max, And Numeric Distributions

In [ ]:
price_numeric_cols = ["open", "high", "low", "close", "adj_close", "volume"]
derived_numeric_cols = ["daily_return", "dollar_change", "intraday_range_pct"]

print("Raw price numeric summary")
display(raw["prices"][price_numeric_cols].describe().T)

print("Clean price numeric summary")
display(clean["prices"][price_numeric_cols + derived_numeric_cols].describe().T)

In [ ]:
clean["prices"][price_numeric_cols].hist(figsize=(12, 8), bins=30);

### Duplicate Counts

In [ ]:
duplicate_checks = [
    ("raw prices", raw["prices"], ["symbol", "date"]),
    ("clean prices", clean["prices"], ["symbol", "date"]),
    ("raw news", raw["news"], ["symbol", "date"]),
    ("clean news", clean["news"], ["symbol", "date"]),
    ("raw filings", raw["filings"], ["symbol", "date", "document_type"]),
    ("clean filings", clean["filings"], ["symbol", "date", "document_type"]),
    ("dashboard", clean["dashboard"], ["symbol", "date"]),
]

pd.DataFrame([
    {
        "dataset": label,
        "key_columns": ", ".join(keys),
        "duplicate_key_rows": int(df.duplicated(keys).sum()),
        "full_duplicate_rows": int(df.duplicated().sum()),
    }
    for label, df, keys in duplicate_checks
])

### Flagged Suspicious Values

In [ ]:
for dataset, df in clean.items():
    flag_cols = [col for col in df.columns if col.startswith("flag_")]
    print(f"\n{dataset}")
    if flag_cols:
        display(df[flag_cols].sum().sort_values(ascending=False))
    else:
        print("No flag columns found.")

### Actual Rows Before And After Transformations

In [ ]:
for name in ["prices", "news", "filings"]:
    print(f"\n{name.upper()} RAW")
    display(raw[name].head(3))
    print(f"{name.upper()} CLEAN")
    display(clean[name].head(3))

### Cleaning Report Review

In [ ]:
report_path = PROCESSED_DIR / "cleaning_report.json"
cleaning_report = json.loads(report_path.read_text())

for name, details in cleaning_report.get("transformations", {}).items():
    print(f"\n{name}")
    print("rows:", details.get("rows_before"), "->", details.get("rows_after"))
    print("columns:", details.get("columns_before"), "->", details.get("columns_after"))
    print("flagged rows:", details.get("flagged_rows"))
    print("removed rows:", details.get("removed_rows"))
    if "merge_match_counts" in details:
        print("merge match counts:", details["merge_match_counts"])

## 7. Assignment 14-Item Checklist Evidence

Use this section as supporting evidence for the PDF checklist. Fill final PDF statuses as `Done`, `Not applicable`, or `Unresolved` after running the notebook from top to bottom.

| # | Checklist Item | Evidence In This Folder |
| --- | --- | --- |
| 1 | State project vision and question | Project docs plus README; cleaning supports the stock/news/filings dashboard. |
| 2 | Define one row | `plan.md`; dashboard and prices use `symbol` + `date`; filings use `symbol` + `date` + `document_type`. |
| 3 | Preserve source | Raw files stay in `data/raw/data`; cleaned files are written separately to `data/processed`. |
| 4 | Inspect columns and types | Type and conversion checks above; conversion rules in `plan.md` and `clean_data.py`. |
| 5 | Standardize text | Before/after row previews and text-length checks above. |
| 6 | Investigate duplicates | Duplicate key checks above. |
| 7 | Handle missing data | Missing-value summaries above. |
| 8 | Check suspicious values | Flag-column summaries above. |
| 9 | Verify merges | Dashboard merge checks and `cleaning_report.json`. |
| 10 | Reconcile changes | Row-count before/after checks and report review above. |
| 11 | Check calculations | Numeric summaries for derived fields above. |
| 12 | Verify individual records | Five-record check table below. |
| 13 | Test reproducibility | Run `python clean_data.py`, then restart and run this notebook. |
| 14 | Document remaining limitations | Real-world representation section below. |


## 8. Five Record Checks

The assignment asks for five record checks with original value, expected result, and actual result. This code creates a reproducible table using representative records from prices, news, filings, and the merged dashboard.

In [ ]:
record_checks = []

# 1. Price record: date conversion and derived return on the second AAPL trading day.
raw_price = raw["prices"].query("symbol == 'AAPL'").sort_values("date").iloc[1]
clean_price = clean["prices"].query("symbol == 'AAPL'").sort_values("date").iloc[1]
record_checks.append({
    "check": "AAPL second price row",
    "original": raw_price[["symbol", "date", "adj_close"]].to_dict(),
    "expected": "symbol remains AAPL, date becomes datetime, daily_return is calculated from previous AAPL adj_close",
    "actual": clean_price[["symbol", "date", "adj_close", "daily_return", "flag_invalid_price"]].to_dict(),
    "matches_expected": bool(clean_price["symbol"] == "AAPL" and pd.notna(clean_price["daily_return"]) and not clean_price["flag_invalid_price"]),
})

# 2. News record: text normalization should preserve meaning and add length.
raw_news_record = raw["news"].iloc[0]
clean_news_record = clean["news"].iloc[0]
record_checks.append({
    "check": "first news text row",
    "original": raw_news_record[["symbol", "date", "highlights"]].to_dict(),
    "expected": "text is HTML/punctuation/whitespace normalized, not rewritten; highlight_length is positive",
    "actual": clean_news_record[["symbol", "date", "highlights", "highlight_length", "flag_empty_text"]].to_dict(),
    "matches_expected": bool(clean_news_record["highlight_length"] > 0 and not clean_news_record["flag_empty_text"]),
})

# 3. Filing record: document type standardization and text-length fields.
raw_filing = raw["filings"].iloc[0]
clean_filing = clean["filings"].iloc[0]
record_checks.append({
    "check": "first filing row",
    "original": raw_filing[["symbol", "date", "document_type"]].to_dict(),
    "expected": "document_type is standardized and text length fields are positive",
    "actual": clean_filing[["symbol", "date", "document_type", "mda_content_length", "risk_content_length", "flag_unknown_document_type"]].to_dict(),
    "matches_expected": bool(clean_filing["document_type"] in ["10-K", "10-Q"] and clean_filing["mda_content_length"] > 0 and clean_filing["risk_content_length"] > 0),
})

# 4. Dashboard filing row: event-based filing should join to a trading-day row when dates match.
dashboard_filing = clean["dashboard"].query("has_filing == True").iloc[0]
record_checks.append({
    "check": "dashboard row with filing",
    "original": "matching cleaned price/news/filing records by symbol and date",
    "expected": "has_filing is True, filing_count is positive, filing_types is populated",
    "actual": dashboard_filing[["symbol", "date", "filing_count", "filing_types", "has_filing"]].to_dict(),
    "matches_expected": bool(dashboard_filing["has_filing"] and dashboard_filing["filing_count"] > 0 and dashboard_filing["filing_types"] != ""),
})

# 5. Dashboard non-filing row: absence of filing should be preserved, not treated as bad data.
dashboard_no_filing = clean["dashboard"].query("has_filing == False").iloc[0]
record_checks.append({
    "check": "dashboard row without filing",
    "original": "trading-day price/news row without same-day filing",
    "expected": "row is preserved, has_filing is False, filing_count is 0",
    "actual": dashboard_no_filing[["symbol", "date", "filing_count", "filing_types", "has_filing"]].to_dict(),
    "matches_expected": bool((not dashboard_no_filing["has_filing"]) and dashboard_no_filing["filing_count"] == 0),
})

pd.DataFrame(record_checks)


## 6. Real-World Representation Check

Ask: **Does the cleaned data still represent the real-world phenomenon correctly?**

Use these questions after reviewing the outputs above:

- Do the stock-price ranges look plausible for these companies and dates?
- Did row counts stay stable, or are removals fully explained?
- Did ticker symbols and filing categories keep their original meaning?
- Are text transformations limited to readability and encoding cleanup?
- Are filings correctly treated as event-based data rather than expected every trading day?
- Do the dashboard joins preserve trading-day price observations even when filings are absent?
- Are any suspicious flags present that require human review before analysis?